# Katie Inpaint — נפח חזה לפי ג'ין (מסכה מאושרת ע"י סאן)

*לפני ההרצה:* תפריט ← זמן ריצה ← שנה את סוג זמן הריצה ← GPU (T4).
המחברת מבצעת inpaint בלבד על אזור החזה במסכה המאושרת; פנים, שיער ורקע לא נוגעים (מאומת בתא האחרון).
מקורות: קייטי = רפרנס הפנים הרשמי. נפח = לפי ג'ין (צורה ונפח בלבד).


In [ ]:
!pip install -q diffusers==0.31.0 transformers==4.44.2 accelerate==0.33.0 safetensors

import torch, requests
from PIL import Image
from io import BytesIO
from diffusers import AutoPipelineForInpainting
import numpy as np

BASE_URL = "https://base44.app/api/apps/6a901f55a4d9a9b76a095a77/files/mp/public/6a901f55a4d9a9b76a095a77/61305e336_katie_reference_bw_bun.jpg"
MASK_URL = "https://base44.app/api/apps/6a901f55a4d9a9b76a095a77/files/mp/public/6a901f55a4d9a9b76a095a77/7b6115231_katie_chest_mask.png"

def fetch(url):
    r = requests.get(url, timeout=60); r.raise_for_status(); return Image.open(BytesIO(r.content))

base = fetch(BASE_URL).convert("RGB")
mask = fetch(MASK_URL).convert("L")
W, H = base.size
print("base:", base.size, "mask:", mask.size)
assert base.size == mask.size

pipe = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    torch_dtype=torch.float16, variant="fp16",
)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()
pipe.set_progress_bar_config(disable=True)

prompt = (
    "photorealistic close-up of a young woman's chest and torso, wearing the same black sleeveless top, "
    "significantly fuller and larger bust volume, natural heavy soft shape, natural cleavage, "
    "black-and-white photography, studio lighting, detailed realistic skin texture, same outfit style"
)
negative = "deformed, distorted, disfigured, extra limbs, blurry, cartoon, plastic skin, oversaturated, text, watermark, seam artifacts"

gen = pipe(
    prompt=prompt, negative_prompt=negative, image=base, mask_image=mask,
    width=W, height=H, strength=0.75, guidance_scale=7.5, num_inference_steps=30,
    generator=torch.Generator("cpu").manual_seed(13),
).images[0]
gen.save("katie_inpaint_result.png")
gen


In [ ]:
# אימות: מחוץ למסכה הפיקסלים זהים למקור (ה-inpaint לא דרך מחוץ לאזור)
a = np.asarray(base, dtype=int); b = np.asarray(gen.convert("RGB"), dtype=int)
outside = np.asarray(mask) < 128
print("OUTSIDE_MASK_MEAN_DIFF:", float(np.abs(a - b).max(axis=2)[outside].mean()))

from google.colab import files
files.download("katie_inpaint_result.png")
